# 第 01 步：我读取一只股票，逐项解释 13 个行情字段

<!-- SNAPSHOT_NOTE -->
## 我的数据准备说明

我在本页记录的是完整快照上的实际运行。如果我从 GitHub 克隆源码，我先在仓库根目录运行 `python publication/restore_snapshot.py`，恢复同版本的 `data/` 与 `runs/`，再执行本步。[完整数据与大型结果索引](../../publication/ASSET_INDEX.md)保留全部路径和哈希；本页图表与 Notebook 已保存输出可直接阅读。


## 我的目标与输入

我先把研究对象从“一个 CSV 文件”变成一张含义清楚的日行情表。本步不计算选股分数，也不模拟买入；我只完成读取、筛选、时间排序、类型核对和第一行解释。我选择 `SYN001`，因为它是固定人工样本中的一个代码，适合在后续收益、动量和回撤课程里持续使用。

我的输入是 `runs/first_demo_20261107/input/prices.csv`。该文件包含 40 只人工股票、73,080 行、13 列。它不是单股文件，因此我不能读取以后直接把相邻两行当作同一只股票的相邻日期。原表按日期与代码组织，同一天先出现多只股票，这是后续最容易产生串股错误的地方。

我已按 [环境准备](../00_environment/README.md) 安装项目和依赖。在仓库根目录执行：

```powershell
.\.venv\Scripts\python.exe examples/01_read_prices.py
```

本步脚本只读取和打印，不会改写原始数据。我在 Notebook 中复现时，需要先找到仓库根目录；脚本里的 `__file__` 不能未经处理直接复制到交互单元，因为交互环境通常没有这个变量。

## 我怎样定位文件

```python
from pathlib import Path
import pandas as pd

project_dir = Path(__file__).resolve().parents[1]
data_path = project_dir / "runs" / "first_demo_20261107" / "input" / "prices.csv"
```

第一行让我用 `Path` 表达文件路径，避免手工拼接不同平台的分隔符。第二行把 pandas 命名为 `pd`，后续表格操作由它完成。第三行先把当前脚本文件解析为绝对位置，再取上两层对应的仓库根目录：脚本在 `examples` 内，`parents[0]` 是 `examples`，`parents[1]` 才是项目根。第四行用路径的 `/` 运算逐段定位输入文件，这里并不是做除法。

我用脚本位置寻找输入，可以减少终端工作目录变化带来的错误。不过，源码中的相对层级仍依赖“脚本放在 examples 内”的结构；移动脚本时，我也要重新检查它。命令中的项目根目录约定与源码内部定位是两个不同层面的规则。

## 我怎样读取并筛选

```python
prices = pd.read_csv(
    data_path,
    dtype={"symbol": "string"},
    parse_dates=["date"],
)
symbol = "SYN001"
stock = prices.loc[prices["symbol"] == symbol].copy()
stock = stock.sort_values("date").reset_index(drop=True)
```

我把 `read_csv` 理解为“按列名读取一张表”。`dtype` 明确要求代码列采用字符串；真实代码 `000001` 若先被读成数字 `1`，前导零就会丢失。`parse_dates` 把日期文本转换为日期类型，之后才能可靠排序、比较和与日历关联。

`prices["symbol"] == symbol` 为每行生成一个 True/False 条件；`.loc[...]` 保留符合条件的整行；`.copy()` 明确把后续操作放在筛出的独立表上。`.loc` 使用方括号，不是函数调用的圆括号。随后 `sort_values("date")` 按时间升序排列，`reset_index(drop=True)` 把行标签重新设为 0、1、2……；`drop=True` 去掉旧索引列，不是删除行情。

我只筛选一只股票后才谈时间序列。这里的“第 0 行”是筛选并排序后的第一条记录，不能混同于股票上市首日；它只是当前文件提供的起点。

## 我实际看到的规模与类型

| 我核对的项目 | 当前结果 |
| --- | --- |
| 全部行情形状 | `(73080, 13)` |
| `SYN001` 形状 | `(1827, 13)` |
| 首个日期 | 2019-01-01 |
| 最后日期 | 2025-12-31 |
| 日期类型 | `datetime64[us]` |
| 代码类型 | `string` |
| 价格、成交量、换手率 | `float64` |
| 三个状态字段 | `bool` |

我的 pandas 版本下日期精度显示为 `us`，不一定是其他版本常见的 `ns`；这不是错误。`shape` 的第一个数字是行数，第二个是列数；它不会告诉我主键是否唯一，因此规模正确仍不替代重复记录检查。`float64` 可以表示小数，也会产生极小的浮点舍入差异；它不表示这些值精确到无限位。

## 我逐项解释第一行

| 字段 | 第一行原值或适度舍入 | 我记录的含义 |
| --- | --- | --- |
| `date` | 2019-01-01 | 这条人工日行情对应的日期 |
| `symbol` | SYN001 | 人工股票标识，不是真实证券代码 |
| `open` | 26.4675166893 | 当日开盘价 |
| `high` | 26.7624864300 | 当日最高价 |
| `low` | 25.9521374713 | 当日最低价 |
| `close` | 26.3273052964 | 当日收盘价 |
| `adj_open` | 26.4675166893 | 复权开盘价；本人工样本没有企业行动，因此等于原价 |
| `adj_close` | 26.3273052964 | 复权收盘价；本样本同样等于原价 |
| `volume` | 3,348,167.8555509425 | 人工成交量代理，本生成器允许小数 |
| `turnover` | 0.016740839277754712 | 换手率比例，约 1.6740839% |
| `can_buy` | True | 模拟执行时允许买入，不是买入信号 |
| `can_sell` | True | 模拟执行时允许卖出，不是已经发生卖出 |
| `status_known` | True | 人工状态由生成器设置，不能解释为真实交易所历史已核验 |

我用基本关系检查 OHLC：最高价不应低于开盘或收盘，最低价不应高于它们。第一行满足 `low ≤ open, close ≤ high`。这些关系可以发现明显的数据错误，却不能证明供应商的每一个价格都正确。

我特别区分原始价和复权价。真实股票发生分红、拆并股等企业行动后，原始报价与连续价格序列的用途不同；当前人工样本没有模拟这些事件，因此两组价格全表相同。后续收益和因子会统一使用 `adj_close`，成交代理使用 `adj_open`，我不能从当前相等关系推断真实数据永远相等。

我还特别记录换手率的单位：`0.0167408393` 是比例，乘 100 后才得到用于阅读的百分数 `1.67408393`。人工生成器将 `turnover` 设置为 `volume / 200_000_000`，因此它不是从真实历史流通股本重新计算的观测。成交量带小数是人工构造的特征，不是一次真实撮合产生小数股的证据。

## 我怎样预览而不误读

```python
preview_columns = ["date", "symbol", "open", "high", "low", "close"]
print(stock[preview_columns].head(5).to_string(index=False))
print(stock.dtypes.to_string())
print(stock.iloc[0].to_string())
```

第一行只选择我此刻关心的列，没有删除其他字段。`head(5)` 限制展示行数，不会缩短 `stock` 本身。`to_string(index=False)` 省去左侧行标签，让行情表更易读；它不改变保存的数据。`dtypes` 逐列显示类型。`.iloc[0]` 按位置取第一行，与 `.loc[某个标签]` 的含义不同。

我看到的前五个日期是 2019-01-01、01-02、01-03、01-04、01-07。周末没有记录，但中国节假日仍可能出现在人工日历中。我因此把它称为“模拟工作日”，没有称为实际 A 股开市安排。

## 我在读取之后补充的质量判断

我把一条行情的业务主键写成“日期与代码的组合”。同一日期出现四十行是正常的，因为股票不同；同一代码在不同日期出现也正常；只有同一日期和同一代码重复，才需要检查是否存在重复下载或来源冲突。我不会看到行数正确便直接通过，也不会用随意保留最后一条的方式掩盖相互矛盾的价格。

我还把类型正确与含义正确分开。价格列成功读成浮点数，不代表价格一定为正、有限，或最高最低关系合法；日期列成功转换，也不代表属于研究日历；代码列是字符串，也不代表证券身份已经核实。后续正式流水线会做更严格检查，我在本节先建立这些问题意识。

对来源的判断，我不只看文件名。我会同时读同目录的 `metadata.json`：当前类型明确是人工数据，随机种子为 20261102，股票总数为四十，价格没有企业行动，状态是人工生成。这些元信息决定我能怎样解释表格。若我把另一份文件复制到相同路径而不保存来源记录，代码仍可能运行，但原来的解释和复现证据已经失效。

我在终端看到科学计数法时，也先换回实际量级再讨论。例如成交量显示为约 `3.348168e+06`，只是一种显示格式，含义仍约为三百三十四万，不是发生了单位转换。小数位减少同样只影响预览；我核对公式时继续使用未舍入的输入。

## 我记录的常见错误与修正

| 我的错误 | 错误原因 | 我的检查方式 |
| --- | --- | --- |
| 我把代码列读成整数 | 真实代码的前导零会丢失 | 我明确使用 `dtype={"symbol": "string"}` |
| 我没有排序就计算下一步收益 | 文件存储顺序未必等于正确时间顺序 | 我筛股后按日期升序排序 |
| 我把 `can_buy=True` 当作策略看好 | 执行许可与预测信号是两类信息 | 我把状态字段与后续得分列分开解释 |
| 我把百分数再次除以 100 | 当前规范化字段已经是比例 | 我先核对来源和数据字典 |
| 我认为首行就是上市日 | 快照起点不等于上市时点 | 我只称它为当前文件首条观测 |
| 我在 Notebook 里直接使用 `__file__` | 交互单元不是以脚本文件方式执行 | 我先显式定位仓库根目录 |

## 我的验收记录与下一步

我已能从原始文件指出单股筛选前后的形状、日期范围、类型和 13 个字段。我保留原数据，没有为了展示而把代码转成数字，也没有把状态字段改成策略信号。本步没有图表产物，表格就是我核对输入的主要证据。

我用一句话总结本步：我已经明确“研究的每一行是什么”，之后才有资格解释“相邻两行之间发生了什么”。下一步是 [第 02 步：简单日收益率](../02_daily_returns/README.md)。

## 我的完整可运行实现

上面的片段用于解释思路；下面各单元按顺序执行完整脚本。我先定位仓库根目录，并显式设置脚本路径，使 `__file__` 在 Notebook 中也有明确含义。

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "pyproject.toml").exists()
                     and (candidate / "examples/01_read_prices.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("我需要从本项目目录或它的子目录打开 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
__file__ = str(PROJECT_ROOT / 'examples/01_read_prices.py')
sys.argv = [__file__]


### 完整实现 · 单元 1

In [2]:
"""第一课：读取一只模拟股票的日行情；只读取文件，不修改原始数据。"""

from pathlib import Path  # 1. 处理文件路径
import pandas as pd  # 2. 导入表格处理库

project_dir = Path(__file__).resolve().parents[1]  # 3. 定位项目根目录
data_path = project_dir / "runs" / "first_demo_20261107" / "input" / "prices.csv"  # 4. 定位输入文件

prices = pd.read_csv(data_path, dtype={"symbol": "string"}, parse_dates=["date"])  # 5. 读取行情
symbol = "SYN001"  # 6. 选择要观察的模拟股票；可以改成 SYN002
stock = prices.loc[prices["symbol"] == symbol].copy()  # 7. 筛选这只股票
stock = stock.sort_values("date").reset_index(drop=True)  # 8. 按时间排序并重排行标签

preview_columns = ["date", "symbol", "open", "high", "low", "close"]  # 9. 选择预览列
print(f"全部行情的形状：{prices.shape}；{symbol} 的形状：{stock.shape}")  # 10. 查看行数和列数
print(stock[preview_columns].head(5).to_string(index=False))  # 11. 预览前 5 行

print("\n每列的数据类型：")
print(stock.dtypes.to_string())  # 12. 查看日期、字符串、数字、布尔类型

print("\n第一行的全部字段：")
print(stock.iloc[0].to_string())  # 13. 逐项查看第一条记录

全部行情的形状：(73080, 13)；SYN001 的形状：(1827, 13)
      date symbol      open      high       low     close
2019-01-01 SYN001 26.467517 26.762486 25.952137 26.327305
2019-01-02 SYN001 26.451148 26.534078 25.435505 25.554247
2019-01-03 SYN001 25.370126 26.501788 25.324328 26.411261
2019-01-04 SYN001 26.392836 26.855037 26.208771 26.475599
2019-01-07 SYN001 26.709078 26.881405 25.805156 26.055965

每列的数据类型：
date            datetime64[us]
symbol                  string
open                   float64
high                   float64
low                    float64
close                  float64
adj_open               float64
adj_close              float64
volume                 float64
turnover               float64
can_buy                   bool
can_sell                  bool
status_known              bool

第一行的全部字段：
date            2019-01-01 00:00:00
symbol                       SYN001
open                      26.467517
high                      26.762486
low                       25.952137
close 

## 我的复核

我在干净内核中按顺序执行以上实现；执行状态记录在仓库的 [Notebook 校验记录](../../publication/notebook_validation.json)。代码中的数据身份、费用和成交假设仍然适用，运行成功只验证对应实现。